In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/yuri/Documents/GitHub/FRE-GY-9743-Assignments-1
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Test interpolation

In [2]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3.0, expected 3.0, diff = 0.0
f(1.0) = 3.0, expected 3.0, diff = 0.0
f(1.5) = 4.0, expected 4.0, diff = 0.0
f(3.0) = 4.0, expected 4.0, diff = 0.0
f(5.5) = 6.0, expected 6.0, diff = 0.0
f(6.5) = 6.0, expected 6.0, diff = 0.0
f(8.0) = 6.0, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [3]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [4]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size                      # bump ordinate i
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(x_s, x_e, this_interp)   # reval
        grad.append((bumped_value - b_value) / bump_size)              # one-sided FD
        values[i] -= bump_size                      # restore

    return np.array(grad)


### Interpolation sensitivity

In [5]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [6]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 7.571721027943568e-12
[1.5, 5.2]: max abs diff = 2.7955415760061442e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 1.1823431123048067e-10


### Additional validation

Two extra checks on the gradients:

1. **Bump-size convergence.** Because $f$ and $I[f]$ are *linear* in the ordinates $y$,
   the one-sided finite difference is exact up to floating-point round-off for *any* bump
   size, so the B&R error should not shrink as the bump shrinks; it should already be at
   machine precision and be dominated by round-off ($\sim \epsilon / h$).
2. **Random knots / random ordinates / reversed limits.** Same comparison on a random
   configuration, including $x_s > x_e$ where $I[f; l, u] = -I[f; u, l]$.

In [7]:
rng = np.random.default_rng(9743)

for h in [1e-2, 1e-4, 1e-6, 1e-8]:
    errs_f = [np.max(np.abs(qfInterpolate1DGrad(x, interp_1d)
                            - bump_reval_interpolator(x, axis1, values, interp_method, extrap_method, h)))
              for x, _ in test_points]
    errs_I = [np.max(np.abs(qfInterpolate1DIntegralGrad(a, b, interp_1d)
                            - bump_reval_interpolator_integrand(a, b, axis1, values, interp_method, extrap_method, h)))
              for (a, b), _ in integration_cases]
    print(f'bump = {h:.0e}:  max|grad f err| = {max(errs_f):.2e},   max|grad I err| = {max(errs_I):.2e}')

print()
rnd_axis = list(np.sort(rng.uniform(-5, 5, size=9)))
rnd_vals = list(rng.normal(size=9))
rnd_interp = qfCreate1DInterpolator(rnd_axis, rnd_vals, interp_method, extrap_method)
worst = 0.0
for _ in range(200):
    a, b = rng.uniform(-8, 8, size=2)          # random limits, either orientation
    g_an = qfInterpolate1DIntegralGrad(a, b, rnd_interp)
    g_br = bump_reval_interpolator_integrand(a, b, rnd_axis, rnd_vals, interp_method, extrap_method)
    worst = max(worst, np.max(np.abs(g_an - g_br)))
    # sanity: analytic integral equals  grad . values  (linearity) and is antisymmetric in the limits
    assert abs(qfInterpolate1DIntegral(a, b, rnd_interp) - g_an @ np.array(rnd_vals)) < 1e-12
    assert abs(qfInterpolate1DIntegral(a, b, rnd_interp) + qfInterpolate1DIntegral(b, a, rnd_interp)) < 1e-12
print(f'random interpolator, 200 random intervals: worst B&R gradient error = {worst:.2e}')

bump = 1e-02:  max|grad f err| = 2.13e-14,   max|grad I err| = 3.98e-13
bump = 1e-04:  max|grad f err| = 2.33e-12,   max|grad I err| = 1.18e-10
bump = 1e-06:  max|grad f err| = 1.40e-10,   max|grad I err| = 5.05e-09
bump = 1e-08:  max|grad f err| = 6.08e-09,   max|grad I err| = 2.97e-07

random interpolator, 200 random intervals: worst B&R gradient error = 1.41e-11


---
# Part I --- Bond Forward Business (written questions)

Notation follows the assignment: the client buys, at $t=0$, a forward on a 10-year US Treasury (maturity $T_m$) settling at $T_s = 1$y. Payoff at settlement

$$V(T_s) = B(T_s; T_s, T_m) - K ,$$

and for $t \in [0, T_s]$

$$V(t) = df_{csa}(t, T_s)\,\bigl(B(t; T_s, T_m) - K\bigr).$$

Throughout, $B(\cdot)$ denotes the **full (dirty) price** per unit of face, $c_i$ the coupon paid at $t_i$, $R$ the term repo rate to $T_s$ (money-market simple rate, ACT/360), $\tau$ the repo accrual factor for $[0, T_s]$, and $df_R(0, t) = 1/(1 + R\,\tau_t)$ the repo discount factor. The contract is for a notional face amount $N = 1{,}000{,}000$ (all formulas below are per unit of face; multiply by $N$).

## Q1. Client motivation, term sheet, settlement flows

**Why would the client buy a 1-year forward on the 10-year UST rather than the bond itself?**

* **Lock in a future purchase price / yield without deploying cash today.** An insurer or pension fund that knows it will receive premiums or contributions in a year, or a portfolio manager expecting a maturity/redemption, can lock in today's forward yield on the asset it intends to buy, removing the risk that yields fall before the cash arrives (classic asset-liability / reinvestment-risk management).
* **Synthetic leverage / balance-sheet efficiency.** A long forward gives essentially the same duration exposure as the bond, but with no cash outlay until $T_s$, no bond on the balance sheet until then and no need to run a repo book. Clients who cannot repo cheaply (or at all) outsource the financing to the dealer.
* **Directional or relative-value view.** The client may simply be bullish on rates (or want to be long the 10-year against something else) and prefers a single OTC contract with a fixed settlement date to rolling futures (no CTD/basis optionality, no daily variation margin if uncollateralised, exact bond and date).
* **Hedging a known future short / issuance.** A corporate treasurer or asset manager who will be short duration in one year can offset it.

**Important term-sheet parameters**

| Parameter | Comment |
|---|---|
| Counterparties, trade date | Bank A (seller / delivers) vs. client (buyer / receives) |
| Underlying security | Exact issue: CUSIP/ISIN, coupon, maturity $T_m$ (on-the-run 10y UST) |
| Notional | Face amount, here 1,000,000 |
| Settlement (forward) date $T_s$ | 1 year; business-day and holiday conventions |
| Forward price $K$ | Quoted **clean**; the accrued interest at $T_s$ and the day-count basis (ACT/ACT for USTs) must be specified so the full settlement amount is unambiguous |
| Settlement type | Physical delivery (DVP of the bond vs. cash) or cash settlement against an agreed fixing (e.g. a screen price / dealer poll at a fixing time) |
| Collateral / CSA terms | Whether the trade is under an ISDA + CSA, thresholds, eligible collateral, the OIS/SOFR discounting curve that defines $df_{csa}$ |
| Calculation agent, fallback and disruption provisions | Who determines the settlement price, what happens if the bond is unavailable / squeezed |
| Early termination / events of default | Standard ISDA provisions |

**What happens on $T_s$?**

* *Physical settlement (the normal case):* Bank A **delivers** the bonds (face $N$) to the client; the client **pays** $N \times \bigl(K_{clean} + AI(T_s)\bigr)$, i.e. the agreed clean forward price plus accrued interest as of $T_s$. Any coupons paid on the bond before $T_s$ belong to Bank A (the bond's owner until delivery), and are already reflected in $K$ (see Q3).
* *Cash settlement:* only the difference changes hands: Bank A pays the client $N\,(B(T_s;T_s,T_m) - K)$ if positive, otherwise the client pays Bank A the absolute value.

## Q2. Why buy the spot bond at inception, and how to fund it

By selling the forward Bank A is **short** the 10-year at $T_s$: its P&L is $-(B(T_s) - K)$ per unit face, i.e. it loses if yields fall. The exposure is large (a 10-year has a DV01 of roughly USD 800 per USD 1mm face, so a 25 bp rally costs about USD 20k on this ticket). Buying the bond today at $B(0;0,T_m)$ neutralises this:

* the desk now owns exactly the bond it has to deliver, so **delivery risk is eliminated**, and
* the package *long spot bond + short forward* has zero exposure to the bond's price, the only remaining cash-flow is the **cost of carrying** the bond from $0$ to $T_s$ (financing cost minus coupons), which is what determines the fair $K$ in Q3.

In other words, the desk does not "price" the forward by forecasting the bond, it **replicates** it: buy the bond, finance it to $T_s$, deliver it. This is the cash-and-carry replication of the forward.

**Funding the position (internal trades)**

The bond costs $N \cdot B(0;0,T_m) \approx 1$mm USD of cash that the rates desk does not have. Two internal legs are booked:

1. **Repo desk (secured funding, the bulk of the cash).** The rates desk enters a **term repo to $T_s$** with the repo desk: it delivers the bond as collateral and receives cash equal to the bond's dirty value less a haircut, agreeing to repay cash plus repo interest at $R$ at $T_s$ and to receive the bond back. The repo is *term* (matched to $T_s$) so the funding rate is locked; rolling an overnight repo would leave the desk exposed to repo-rate moves (see Q5). Economically the desk has borrowed cash at the bond's specific repo rate, which is the cheapest secured funding available for a Treasury.
2. **Treasury (funding) desk (unsecured, the residual).** The haircut on the repo (say 2%) plus any initial margin / variation margin needs and the cost of the desk's own capital are funded unsecured from the internal treasury desk at the bank's internal **funds transfer price (FTP)**. The treasury desk also pays/charges the desk for the CSA collateral posted or received against the client forward.

At $T_s$ the repo unwinds: the desk repays cash + interest, gets the bond back and delivers it to the client against $N(K_{clean}+AI)$. If $K$ was set as in Q3 these cash-flows net to zero (plus the margin discussed in Q5).

## Q3. Computing $B(0; T_s, T_m)$ from the spot price and the term repo rate

Follow the money in the cash-and-carry package of Q2 (per unit of face).

1. **Hint 1, cost of borrowing.** Borrowing 1 dollar from the repo desk for $\tau$ years at simple rate $R$ costs $1 + R\tau$ at $T_s$. So borrowing the full purchase price $B(0;0,T_m)$ means repaying $B(0;0,T_m)\,(1 + R\tau)$ at $T_s$. Equivalently, the repo discount factor is $df_R(0,T_s) = 1/(1+R\tau)$.

2. **Hint 2, coupons reduce the debt.** While the bond sits as collateral, any coupon $c_i$ paid at $t_i \le T_s$ is paid to the repo desk and is applied against the outstanding loan, so from $t_i$ onwards the desk owes $c_i$ less principal (and hence less interest). A coupon of $c_i$ received at $t_i$ is worth $c_i (1 + R\,\tau_{t_i,T_s})$ at $T_s$ or, equivalently, it reduces the *time-0* amount that has to be borrowed by its repo-discounted value $c_i\,df_R(0,t_i)$.

3. **Zero net cost at $T_s$.** The forward price must make the package cost nothing: the cash received from the client at $T_s$ exactly repays the repo. Hence

$$\boxed{\;B(0;T_s,T_m) \;=\; \frac{B(0;0,T_m) \;-\; \displaystyle\sum_{0 < t_i \le T_s} c_i\, df_R(0,t_i)}{df_R(0,T_s)}
\;=\; \Bigl(B(0;0,T_m) - \sum_{0 < t_i \le T_s}\frac{c_i}{1+R\,\tau_{t_i}}\Bigr)(1+R\tau)\;}$$

with $\tau_{t_i}$ the accrual factor from $0$ to the coupon date. This is the **forward dirty price**, "spot minus carry": the spot price grossed up by the financing cost and reduced by the coupon income earned while the bond is held. The **clean** forward price quoted on the term sheet is $K_{clean} = B(0;T_s,T_m) - AI(T_s)$, with $AI(T_s)$ the accrued interest on the bond at the settlement date. (If one only has a single term rate $R$ it is used to discount the coupons as well; with a repo curve one would use the matching term rates. The alternative bookkeeping "coupon reinvested at $R$ from $t_i$ to $T_s$", $B(0;0,T_m)(1+R\tau) - \sum c_i(1+R\,\tau_{t_i,T_s})$, differs only at order $R^2$, i.e. by a fraction of a cent per 100 face.)

The inputs are exactly the three listed in the question: the coupon schedule (dates $t_i$ and amounts $c_i$ falling in $(0, T_s]$), the term repo rate $R$, and the spot dirty price $B(0;0,T_m)$. A numerical example is worked out in the code cell below.

## Q4. Mark-to-market at $t \in (0, T_s]$

Apply the same logic starting from $t$ instead of $0$. Re-observe the market and recompute the *current* forward price with the remaining coupons and the *current* term repo rate $R_t$ for $[t, T_s]$:

$$B(t;T_s,T_m) = \frac{B(t;t,T_m) - \displaystyle\sum_{t < t_i \le T_s} c_i\, df_{R_t}(t,t_i)}{df_{R_t}(t,T_s)},$$

then discount the locked-in difference to today with the CSA discount factor, which comes from the OIS/SOFR curve implied by the CSA collateral terms (not the repo rate, because the forward's MTM is collateralised at the CSA rate):

$$V(t) = N \cdot df_{csa}(t,T_s)\,\bigl(B(t;T_s,T_m) - K\bigr) \qquad\text{(from the client's, i.e. the buyer's, point of view; Bank A's value is } -V(t)).$$

At $t = T_s$, $df_{csa} = 1$ and the sum is empty, recovering the payoff $B(T_s;T_s,T_m) - K$. Note that the MTM of the forward alone moves with the bond price; the MTM of the *hedged book* (forward + spot bond + term repo) is essentially flat, only the small residuals discussed in Q5 remain. In practice the desk risk-manages $V(t)$ by bumping its inputs: the spot bond price (or yield), the term repo rate and the OIS curve, which is exactly the kind of gradient computation that Part II of this homework builds a component for.

## Q5. Residual risk, and how the desk makes money

**Does the hedged desk have market risk?** Following Q2 and Q3 with a *term* repo, the desk's cash-flows are fully determined at inception: buy bond, lock the financing to $T_s$ at $R$, deliver at $K$. **Outright rate (duration) risk is zero.** What is left is second order:

* *Repo / funding risk* if the financing is not fully termed out (rolling overnight or short-dated repo exposes the desk to repo-rate moves and to the bond going "special" or "off special"), plus the unsecured funding of the haircut at FTP.
* *Counterparty credit risk* on the client: if the client defaults when $B(T_s) < K$ the desk is left long a bond that is worth less than its repo debt; mitigated by the CSA (collateral posted daily) and priced as CVA.
* *Basis between CSA discounting and repo funding*, and coupon-reinvestment risk (small).
* *Operational / delivery risk* (fails, settlement mismatches).

**No risk means no P&L, so where does revenue come from?** The desk quotes the client a forward price $K$ that is *higher* than the fair replication price of Q3. The cleanest way to see (and explain) this is through the repo rate, as the hint suggests. The forward price is increasing in the repo rate,

$$\frac{\partial B(0;T_s,T_m)}{\partial R} \approx B(0;0,T_m)\,\tau \;>\; 0,$$

so quoting the client a forward price computed with an **implied repo rate $R_{quote} = R + s$** (i.e. charging the client a financing spread $s$) yields a locked-in margin of roughly

$$\text{Revenue} \;\approx\; N \cdot B(0;0,T_m)\,\tau\, s ,$$

e.g. about USD 1,000 per 10 bp of spread on a USD 1mm 1-year forward (see the code cell). Because the desk actually finances at $R$ and the client pays $K$ computed at $R+s$, this spread is earned with certainty (up to the residual risks above), it is carry, not a bet on rates.

**How to convince the client.** The pitch is a financing comparison, not a rates view:

* The client's alternative is to buy the bond today and fund it itself. A real-money client either cannot repo at all or repos at a worse rate than a primary dealer (no netting benefits, lower balance-sheet efficiency, GC rather than special). As long as $R + s$ is below the client's **own** all-in funding cost, the forward is cheaper than doing it themselves, so both sides gain.
* Operational convenience: one confirm, no daily repo rolls, no collateral management on the bond, no balance-sheet usage until $T_s$, exact bond and date (no futures basis / CTD risk).
* If the bond trades **special** (repo rate below GC because the issue is in demand), the dealer's true financing cost $R$ is very low; the dealer can quote an implied repo closer to GC, which still looks attractive to the client relative to any rate the client can achieve, while the desk captures the specialness. The 10-year on-the-run frequently trades special, which is precisely why this trade is attractive for a dealer.
* Transparency: the client can verify the quote from screen prices, the spot price and a published repo rate, so the spread $s$ is small and defensible; revenue comes from volume and from financing access, not from an information edge.

## *Q5. Relation to risk-neutral pricing

The forward price obtained in Q3 by replication is exactly the risk-neutral (change-of-numeraire) forward price under a few explicit assumptions.

Take the $T_s$-maturity zero-coupon bond $P(t,T_s)$ as numeraire. Under the associated $T_s$-forward measure $\mathbb{Q}^{T_s}$ the price of any traded asset divided by the numeraire is a martingale, so for the (ex-coupon) bond

$$B(t;T_s,T_m) := \mathbb{E}^{\mathbb{Q}^{T_s}}_t\bigl[B(T_s;T_s,T_m)\bigr] = \frac{B(t;t,T_m) - \sum_{t<t_i\le T_s} c_i\,P(t,t_i)}{P(t,T_s)},$$

because the value today of "the bond delivered at $T_s$" equals the value of the bond stripped of the coupons paid before $T_s$. The forward contract itself then has value

$$V(t) = P(t,T_s)\,\mathbb{E}^{\mathbb{Q}^{T_s}}_t\bigl[B(T_s;T_s,T_m) - K\bigr] = P(t,T_s)\bigl(B(t;T_s,T_m) - K\bigr),$$

which is equation (2) of the assignment with $df_{csa}(t,T_s) = P(t,T_s)$, and $K$ is set so that $V(0)=0$, i.e. $K = B(0;T_s,T_m)$.

Comparing with Q3, the two formulas coincide if and only if

1. **the repo discount factors are the risk-free discount factors**, $df_R(0,t) = P(0,t)$, i.e. the term repo rate equals the (OIS/CSA) risk-free rate for every maturity up to $T_s$: no repo/OIS basis, no specialness, no haircut that needs unsecured funding, and a full repo curve (or a flat $R$) used consistently for the coupons and for $T_s$;
2. **coupons are deterministic** (true for a fixed-coupon Treasury) and are received/passed through with certainty, and the bond can be bought, repoed and delivered without frictions (no bid-offer, no fails, no credit risk between the desks or with the client);
3. **the CSA discounting equals the numeraire**, so that (2) holds with $df_{csa} = P$.

Under these assumptions the trader's "spot plus carry" and the quant's "expectation under the $T_s$-forward measure" are the same computation: the replication argument *is* the proof of the martingale property, and the forward price is model-independent because it depends only on today's spot price and the discount curve, not on any assumption about bond-price dynamics or volatility. When the assumptions fail (repo trades away from OIS, the bond goes special, the client is uncollateralised), the differences show up precisely as the basis and funding adjustments discussed in Q5, and the desk quotes them as a spread to the implied repo rate.

### Numerical illustration of Q3 -- Q5

A stylised on-the-run 10-year UST: 4.25% semi-annual coupon, next coupons in 0.35y and 0.85y,
clean price 99.00 (so about 0.64 of accrued), term repo 4.30% ACT/360 to $T_s = 1$y.
All prices per 100 face; the trade is for 1,000,000 face.

In [8]:
N          = 1_000_000          # face amount of the forward
coupon     = 4.25               # annual coupon, % of face, paid semi-annually
c          = coupon / 2         # coupon per period, per 100 face
cpn_dates  = [0.35, 0.85]       # coupon dates (years from today) falling in (0, T_s]
Ts         = 1.0                # forward settlement (years)
tau        = 365 / 360          # ACT/360 accrual factor for the 1y repo
R          = 0.0430             # term repo rate to T_s
clean_spot = 99.00
ai_0       = c * (0.5 - 0.35) / 0.5          # accrued today (0.15y since last coupon, linear ACT/ACT proxy)
dirty_spot = clean_spot + ai_0
ai_Ts      = c * (Ts - 0.85) / 0.5           # accrued at T_s (0.15y after the 0.85y coupon)

def df_repo(t, R=R):
    return 1.0 / (1.0 + R * t * 365 / 360)

def forward_dirty(dirty_spot, R):
    pv_coupons = sum(c * df_repo(t, R) for t in cpn_dates)
    return (dirty_spot - pv_coupons) / df_repo(Ts, R)

F_dirty = forward_dirty(dirty_spot, R)
K_clean = F_dirty - ai_Ts
print(f'spot: clean {clean_spot:.4f}, accrued {ai_0:.4f}, dirty {dirty_spot:.4f}')
print(f'PV(coupons to T_s) at repo   = {sum(c*df_repo(t) for t in cpn_dates):.4f}')
print(f'forward DIRTY price B(0;Ts,Tm) = {F_dirty:.4f}')
print(f'forward CLEAN price K          = {K_clean:.4f}   (accrued at T_s = {ai_Ts:.4f})')
print(f'client pays at T_s: N*(K+AI)/100 = ${N*(K_clean+ai_Ts)/100:,.2f}  and receives {N:,} face of the bond')

# alternative bookkeeping: coupons reinvested at R from t_i to T_s -- same to O(R^2)
F_alt = dirty_spot*(1+R*tau) - sum(c*(1+R*(Ts-t)*365/360) for t in cpn_dates)
print(f'\n"coupon reinvested at R" variant  = {F_alt:.4f}   (difference {F_dirty-F_alt:+.5f} per 100)')

# --- Q4: MTM at t = 0.5y after a 25bp rally (price up ~2 pts) and repo unchanged, OIS = repo for simplicity
t = 0.5
dirty_t   = 101.10 + c*(t-0.35)/0.5          # new clean 101.10 plus accrued since the 0.35y coupon
remaining = [d for d in cpn_dates if d > t]
F_t = (dirty_t - sum(c/(1+R*(d-t)*365/360) for d in remaining)) * (1 + R*(Ts-t)*365/360)
df_csa = 1/(1+0.0425*(Ts-t)*365/360)         # CSA (SOFR OIS) discount factor t -> T_s
V_t = N/100 * df_csa * (F_t - F_dirty)
print(f'\nQ4: at t={t}y forward dirty price = {F_t:.4f}; MTM to client = ${V_t:,.2f}  (bank A: ${-V_t:,.2f})')

# --- Q5: revenue from quoting an implied repo R + s
for s_bp in [5, 10, 25]:
    K_quote = forward_dirty(dirty_spot, R + s_bp/1e4)
    print(f'Q5: implied repo R+{s_bp:>2}bp -> K = {K_quote-ai_Ts:.4f} clean, '
          f'locked-in revenue = ${N/100*(K_quote-F_dirty):,.2f}  '
          f'(approx N*B*tau*s = ${N/100*dirty_spot*tau*s_bp/1e4:,.2f})')

spot: clean 99.0000, accrued 0.6375, dirty 99.6375
PV(coupons to T_s) at repo   = 4.1421
forward DIRTY price B(0;Ts,Tm) = 99.6587
forward CLEAN price K          = 99.0212   (accrued at T_s = 0.6375)
client pays at T_s: N*(K+AI)/100 = $996,587.05  and receives 1,000,000 face of the bond

"coupon reinvested at R" variant  = 99.6573   (difference +0.00140 per 100)

Q4: at t=0.5y forward dirty price = 101.8165; MTM to client = $21,123.33  (bank A: $-21,123.33)
Q5: implied repo R+ 5bp -> K = 99.0709 clean, locked-in revenue = $496.81  (approx N*B*tau*s = $505.11)
Q5: implied repo R+10bp -> K = 99.1206 clean, locked-in revenue = $993.63  (approx N*B*tau*s = $1,010.21)
Q5: implied repo R+25bp -> K = 99.2696 clean, locked-in revenue = $2,484.10  (approx N*B*tau*s = $2,525.53)
